# End-of-Season Retro & Model Update

Three questions this notebook exists to answer, in order:

1. **Did the money come from the model, or from variance?** Realized P&L, decomposed, with
   interval estimates that respect the fact that bets on the same slate are correlated.
2. **Where is the model actually calibrated, and where is it not?** Brier decomposition and
   per-bucket reliability, on the season we just bet.
3. **What changes for next season?** Retrain on the rolled window, evaluate the new defensive
   shot-profile feature, and re-derive (not re-assume) the edge zones.

Sections 6 (bankroll) and 7 (CLV) are the two that most often overturn conclusions from
sections 3-5. Do not skip them.

## 0. Config

In [1]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)


In [2]:
# Odds conversion helpers. Kept as functions because every downstream section needs them.

def american_to_decimal(odds):
    odds = np.asarray(odds, dtype=float)
    return np.where(odds > 0, 1 + odds / 100.0, 1 + 100.0 / np.abs(odds))


def american_to_implied(odds):
    return 1.0 / american_to_decimal(odds)


def devig_multiplicative(p_over, p_under):
    # Proportional (multiplicative) devig. Simplest defensible choice.
    # Alternative worth testing in section 7: Shin, or additive. On two-way markets in the
    # +140/+200 range the three methods disagree by roughly 0.5-1.5pp of fair probability,
    # which is a meaningful fraction of a 5% edge. Do not treat this as a free choice.
    total = p_over + p_under
    return p_over / total, p_under / total

## 1. Load model predictions for the season

## 2. Load realized bets (Pikkit) and join to predictions

This join is the load-bearing step of the whole retro. Two failure modes to watch:

- **Player identity.** Pikkit exports a display-name string; predictions key on `player_id`.
  Suffixes, accents, and nicknames break naive matching.
- **Date.** Pikkit records *placement* timestamp in local time; predictions key on slate date.
  A bet placed at 11:40pm ET for a late West-coast game will land on the wrong slate date if
  you just truncate the timestamp.

Track the match rate explicitly. A silent 90% match rate means the 10% you dropped is
non-random (late bets, live bets, traded players) and every ROI number below is biased.

In [ ]:
bets = pd.read_csv(PIKKIT_CSV)
print(bets.columns.tolist())
bets.head()

In [ ]:
# Normalize. Adjust column names to match your export schema.
bets = bets.rename(columns={
    'Bet Placed': 'placed_at',
    'Sportsbook': 'book',
    'Odds': 'odds',
    'Stake': 'stake',
    'Profit': 'profit',
    'Status': 'status',
})

bets['placed_at'] = pd.to_datetime(bets['placed_at'])
# Slate date = placement date shifted back 6h so post-midnight placements attach correctly.
bets['date'] = (bets['placed_at'] - pd.Timedelta(hours=6)).dt.normalize()

bets = bets[bets['status'].isin(['won', 'lost', 'push'])].copy()
print(bets.shape)

In [ ]:
# Parse the market string into (player, over_under, number).
# TODO: adapt regex to your export format, e.g. 'Stephen Curry Over 3.5 Three Pointers Made'
pattern = r'^(?P<player_name>.+?)\s+(?P<over_under>Over|Under)\s+(?P<number>[\d.]+)'
parsed = bets['Bet Description'].str.extract(pattern)

bets = pd.concat([bets, parsed], axis=1)
bets['over_under'] = bets['over_under'].str.lower()
bets['number'] = bets['number'].astype(float)

print('unparsed rows: {}'.format(bets['player_name'].isna().sum()))
bets[bets['player_name'].isna()].head(20)

In [ ]:
# Name -> player_id. Exact match first, then inspect the residue by hand.
names = pd.read_sql('SELECT player_id, name FROM players', con)
names['join_name'] = names['name'].str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('ascii').str.lower().str.strip()
bets['join_name'] = bets['player_name'].str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('ascii').str.lower().str.strip()

bets = bets.merge(names[['player_id', 'join_name']], on='join_name', how='left')
print('unmatched names: {}'.format(bets['player_id'].isna().sum()))
bets.loc[bets['player_id'].isna(), 'player_name'].value_counts()

In [ ]:
# The join. Keep it a LEFT join from bets so unmatched bets stay visible.
df = bets.merge(
    preds,
    on=['player_id', 'date', 'over_under', 'number'],
    how='left',
    indicator=True,
)

match_rate = (df['_merge'] == 'both').mean()
print('match rate: {:.1%}  ({} of {} bets)'.format(match_rate, (df['_merge'] == 'both').sum(), df.shape[0]))

In [ ]:
# Audit the misses before proceeding. Look for structure: one book, one date range,
# one class of player. Structure means bias, not noise.
misses = df[df['_merge'] == 'left_only']
print(misses.groupby('book').size())
print(misses.groupby(misses['date'].dt.to_period('M')).size())
print(misses['number'].value_counts().head(10))

In [ ]:
matched = df[df['_merge'] == 'both'].drop(columns=['_merge']).copy()

matched['dec_odds'] = american_to_decimal(matched['odds'])
matched['implied'] = 1.0 / matched['dec_odds']
matched['edge'] = matched['model_prob'] - matched['implied']
matched['kelly_full'] = (matched['model_prob'] * matched['dec_odds'] - 1) / (matched['dec_odds'] - 1)
matched['kelly_frac'] = matched['kelly_full'] * KELLY_FRACTION

matched.shape

## 3. Realized P&L

### 3a. Three different numbers all called "ROI"

Compute all three. They will not agree, and the gap between them is itself a finding.

- **Turnover ROI** = total profit / total staked. The number a book would quote.
- **Mean daily ROI** = average of (daily profit / daily stake). This is the one that tends to
  get reported, and it is *upward biased* relative to turnover ROI whenever small-stake days
  happen to be good days, because it weights every day equally regardless of capital at risk.
- **Bankroll CAGR** = geometric growth of the bankroll. The only one that answers "how much
  richer am I."

If the 8.5% figure is the middle one, expect the first to be lower and the third to be lower
still. Decide now which one is the headline metric for next season and stick to it.

In [ ]:
daily = matched.groupby('date').agg(stake=('stake', 'sum'), profit=('profit', 'sum'))
daily['roi'] = daily['profit'] / daily['stake']

turnover_roi = matched['profit'].sum() / matched['stake'].sum()
mean_daily_roi = daily['roi'].mean()

print('bets:              {}'.format(matched.shape[0]))
print('total staked:      {:,.0f}'.format(matched['stake'].sum()))
print('total profit:      {:,.0f}'.format(matched['profit'].sum()))
print('turnover ROI:      {:.2%}'.format(turnover_roi))
print('mean daily ROI:    {:.2%}'.format(mean_daily_roi))
print('median daily ROI:  {:.2%}'.format(daily['roi'].median()))

In [ ]:
daily['cum_profit'] = daily['profit'].cumsum()
fig, ax = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
daily['cum_profit'].plot(ax=ax[0], title='Cumulative profit')
daily['stake'].plot(ax=ax[1], title='Daily stake')
plt.tight_layout()
plt.show()

### 3b. Slice by the dimensions you already believe matter

The prior going in: edge concentrated in unders in the ~25-61% implied range, plus longshot
overs; overs above ~50% implied and unders below ~25% implied are negative-ROI. The +140 to
+200 sweet spot is roughly 33-42% implied.

Treat this section as a **test of a pre-registered hypothesis**, not as a search. Every extra
cut you take multiplies the number of implicit comparisons and inflates the chance that some
bucket looks great by luck.

In [ ]:
IMPLIED_BINS = [0, 0.15, 0.25, 0.33, 0.42, 0.50, 0.61, 0.75, 1.0]
matched['implied_bucket'] = pd.cut(matched['implied'], IMPLIED_BINS)


def summarize(g):
    return pd.Series({
        'n': len(g),
        'stake': g['stake'].sum(),
        'profit': g['profit'].sum(),
        'roi': g['profit'].sum() / g['stake'].sum(),
        'win_rate': (g['profit'] > 0).mean(),
        'mean_edge': g['edge'].mean(),
    })


print(matched.groupby(['over_under', 'implied_bucket']).apply(summarize))

In [ ]:
for dim in ['book', 'number', 'over_under']:
    print('--- {} ---'.format(dim))
    print(matched.groupby(dim).apply(summarize))
    print()

In [ ]:
matched['month'] = matched['date'].dt.to_period('M')
print(matched.groupby('month').apply(summarize))

In [ ]:
# ROI by model edge decile. If the model is well calibrated, ROI should rise monotonically.
# It almost certainly will not. Where it breaks is where the PMF is overconfident.
matched['edge_decile'] = pd.qcut(matched['edge'], 10, duplicates='drop')
print(matched.groupby('edge_decile').apply(summarize))

## 4. Is any of this signal?

Bets on the same slate share opponent, pace, and referee conditions, and often share a game.
Treating them as independent understates the variance of ROI substantially. Block bootstrap by
slate date is the minimum defensible correction.

Second question this section answers: **how many bets would you need** to distinguish 8.5% ROI
from 0% at conventional confidence? At +170 (dec 2.70), per-bet profit has a standard deviation
near 1.1 units of stake. To get a standard error of ~2% on ROI you need roughly (1.1/0.02)^2
which is about 3,000 *independent* bets. With slate clustering the effective count is lower than
the raw count. Run the numbers below rather than trusting that arithmetic.

In [ ]:
def block_bootstrap_roi(frame, n_boot=5000, seed=0):
    rng = np.random.default_rng(seed)
    dates = frame['date'].unique()
    by_date = {d: g for d, g in frame.groupby('date')}
    out = np.empty(n_boot)
    for i in range(n_boot):
        draw = rng.choice(dates, size=len(dates), replace=True)
        stake = 0.0
        profit = 0.0
        for d in draw:
            g = by_date[d]
            stake += g['stake'].sum()
            profit += g['profit'].sum()
        out[i] = profit / stake
    return out


boot = block_bootstrap_roi(matched)
print('point ROI:  {:.2%}'.format(turnover_roi))
print('95% CI:     [{:.2%}, {:.2%}]'.format(np.percentile(boot, 2.5), np.percentile(boot, 97.5)))
print('P(ROI > 0): {:.1%}'.format((boot > 0).mean()))

plt.figure(figsize=(10, 3))
plt.hist(boot, bins=60)
plt.axvline(0, color='k')
plt.title('Block-bootstrapped season ROI')
plt.show()

In [ ]:
# Same treatment for the pre-registered edge zones. Wide intervals here are the expected
# result, not a failure. The point is to know how wide.
rows = []
for (ou, bucket), g in matched.groupby(['over_under', 'implied_bucket']):
    if len(g) < 30:
        continue
    b = block_bootstrap_roi(g, n_boot=2000)
    rows.append({
        'over_under': ou,
        'bucket': str(bucket),
        'n': len(g),
        'roi': g['profit'].sum() / g['stake'].sum(),
        'lo': np.percentile(b, 2.5),
        'hi': np.percentile(b, 97.5),
        'p_positive': (b > 0).mean(),
    })

zone_ci = pd.DataFrame(rows).sort_values('roi', ascending=False)
zone_ci

In [ ]:
# Effective sample size given clustering: compare naive SE to block-bootstrap SE.
naive_se = matched['profit'].std() / np.sqrt(len(matched)) / matched['stake'].mean()
block_se = boot.std()
print('naive SE on ROI:     {:.3%}'.format(naive_se))
print('block SE on ROI:     {:.3%}'.format(block_se))
print('design effect:       {:.2f}'.format((block_se / naive_se) ** 2))
print('effective n:         {:.0f}'.format(len(matched) / ((block_se / naive_se) ** 2)))

## 5. Model calibration on the season just played

Betting ROI and calibration are different questions. You can profit with a miscalibrated model
if the books are miscalibrated in the same direction but further. Measure both.

In [ ]:
actual_q = '''
SELECT player_id, game_date AS date, threesMade
FROM pgames
WHERE game_date BETWEEN '{}' AND '{}'
'''.format(SEASON_START, SEASON_END)

actual = pd.read_sql(actual_q, con)
actual['date'] = pd.to_datetime(actual['date'])
actual['threesMade'] = actual['threesMade'].clip(upper=10)

In [ ]:
# Full PMF for the season. Adjust to however you persist the 11-class output.
pmf_q = '''
SELECT player_id, date, {}
FROM pmf_predictions
WHERE date BETWEEN '{}' AND '{}'
'''.format(', '.join(['p{}'.format(k) for k in range(11)]), SEASON_START, SEASON_END)

pmf = pd.read_sql(pmf_q, con)
pmf['date'] = pd.to_datetime(pmf['date'])
pmf = pmf.merge(actual, on=['player_id', 'date'], how='inner')

P = pmf[['p{}'.format(k) for k in range(11)]].to_numpy()
y = pmf['threesMade'].to_numpy().astype(int)
Y = np.zeros_like(P)
Y[np.arange(len(y)), y] = 1

brier = ((P - Y) ** 2).sum(axis=1).mean()
logloss = -np.log(np.clip(P[np.arange(len(y)), y], 1e-12, None)).mean()
print('multiclass Brier: {:.4f}'.format(brier))
print('log loss:         {:.4f}'.format(logloss))
print('n player-games:   {}'.format(len(y)))

In [ ]:
# Brier by outcome class. This is where the 1-2 and 8-9 tail overconfidence should show up.
rows = []
for k in range(11):
    rows.append({
        'made': k,
        'n_actual': int((y == k).sum()),
        'mean_pred': P[:, k].mean(),
        'base_rate': (y == k).mean(),
        'component_brier': ((P[:, k] - Y[:, k]) ** 2).mean(),
    })
per_class = pd.DataFrame(rows)
per_class['pred_minus_actual'] = per_class['mean_pred'] - per_class['base_rate']
per_class

In [ ]:
# Reliability curve per class. Positive gap = overconfident at that outcome.
fig, axes = plt.subplots(3, 4, figsize=(16, 10), sharex=True, sharey=True)
for k, ax in zip(range(11), axes.ravel()):
    bins = pd.qcut(P[:, k], 10, duplicates='drop')
    tmp = pd.DataFrame({'p': P[:, k], 'y': Y[:, k], 'bin': bins})
    agg = tmp.groupby('bin').agg(pred=('p', 'mean'), obs=('y', 'mean'))
    ax.plot([0, 1], [0, 1], 'k--', lw=0.8)
    ax.plot(agg['pred'], agg['obs'], 'o-')
    ax.set_title('{} made'.format(k))
axes.ravel()[-1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Calibration restricted to the bets you actually placed. This is the money-relevant view:
# the model only has to be calibrated on the tail probability implied by each line.
bins = pd.qcut(matched['model_prob'], 10, duplicates='drop')
cal = matched.assign(bin=bins, hit=(matched['profit'] > 0).astype(int)).groupby('bin').agg(
    n=('hit', 'size'), pred=('model_prob', 'mean'), obs=('hit', 'mean'), implied=('implied', 'mean')
)
cal['model_error'] = cal['pred'] - cal['obs']
cal['book_error'] = cal['implied'] - cal['obs']
cal

In [ ]:
# Prior-season Brier for comparison. Pull from archive DB if seasons have rolled off.
# Fill in from stored evaluation artifacts rather than recomputing where possible.
brier_history = pd.DataFrame({
    'season': PRIOR_SEASONS + [SEASON],
    'brier': [np.nan, np.nan, brier],
})
brier_history

## 6. Bankroll and Kelly diagnostics

Three things to test here, in increasing order of how uncomfortable the answer may be.

**6a. How large did bets actually get?** Fractional Kelly at 1/8 against a static bankroll
should cap out well under 2% of bankroll. Where it did not, the model's probability was far
enough from the book's that Kelly sized aggressively. Those bets going well is *consistent with*
edge and *also* consistent with luck; the sample of them is small by construction.

**6b. Static vs. dynamic bankroll.** A static denominator means you underbet after gains and
overbet after losses, relative to Kelly's own logic. It also caps absolute drawdown and removes
path-dependence from sizing. Both counterfactuals are computable; compute them.

**6c. Path dependence.** Terminal bankroll under any dynamic scheme depends on the order the
bets arrived in. Resample orderings to get a distribution rather than a single number.

In [ ]:
matched['pct_of_bankroll'] = matched['stake'] / STARTING_BANKROLL

print(matched['pct_of_bankroll'].describe(percentiles=[0.5, 0.9, 0.99]))
print()
print('bets over 2% of bankroll: {}'.format((matched['pct_of_bankroll'] > 0.02).sum()))

big = matched[matched['pct_of_bankroll'] > 0.02]
print(summarize(big) if len(big) else 'none')

In [ ]:
# The uncomfortable cut: performance of the outsized bets specifically, with an interval.
if len(big) >= 30:
    b = block_bootstrap_roi(big, n_boot=3000)
    print('outsized-bet ROI: {:.2%}  CI [{:.2%}, {:.2%}]'.format(
        big['profit'].sum() / big['stake'].sum(),
        np.percentile(b, 2.5), np.percentile(b, 97.5)))
else:
    print('n = {}, too few to say anything'.format(len(big)))

In [ ]:
def replay(frame, mode='static', bankroll=None, kelly=KELLY_FRACTION, cap=None, order=None):
    # mode: 'static' | 'dynamic'
    # cap: max fraction of bankroll per bet, or None
    bankroll = STARTING_BANKROLL if bankroll is None else bankroll
    f = frame if order is None else frame.iloc[order]
    bk = bankroll
    path = []
    for _, r in f.iterrows():
        base = bankroll if mode == 'static' else bk
        frac = max(r['kelly_full'] * kelly, 0.0)
        if cap is not None:
            frac = min(frac, cap)
        stake = base * frac
        won = r['profit'] > 0
        bk += stake * (r['dec_odds'] - 1) if won else -stake
        path.append(bk)
    return np.array(path)


ordered = matched.sort_values('date')
paths = {
    'static':          replay(ordered, mode='static'),
    'dynamic':         replay(ordered, mode='dynamic'),
    'dynamic_cap_2pct': replay(ordered, mode='dynamic', cap=0.02),
    'static_cap_2pct':  replay(ordered, mode='static', cap=0.02),
}

plt.figure(figsize=(14, 5))
for k, v in paths.items():
    plt.plot(v, label=k)
plt.legend()
plt.title('Bankroll under alternative sizing rules')
plt.show()

for k, v in paths.items():
    dd = (np.maximum.accumulate(v) - v) / np.maximum.accumulate(v)
    print('{:20s} terminal {:>12,.0f}   max DD {:.1%}'.format(k, v[-1], dd.max()))

In [ ]:
# Order sensitivity. If the spread across shuffles is wide, the single realized path tells
# you much less than it appears to.
rng = np.random.default_rng(0)
terminals = []
for _ in range(300):
    order = rng.permutation(len(ordered))
    terminals.append(replay(ordered, mode='dynamic', order=order)[-1])

terminals = np.array(terminals)
print('realized order terminal: {:,.0f}'.format(paths['dynamic'][-1]))
print('shuffled: median {:,.0f}   5-95pct [{:,.0f}, {:,.0f}]'.format(
    np.median(terminals), np.percentile(terminals, 5), np.percentile(terminals, 95)))

## 7. Closing line value

CLV is the single best available proxy for whether the edge is real, because it is measured
against a much larger sample of market prices than your own bet outcomes. A model with genuine
edge should beat the close on average even in a season where the bets themselves lost.

If ROI is positive but CLV is flat, the most likely explanation is that you got lucky. If CLV is
positive but ROI is flat, the edge is probably real and the sample is too small.

In [ ]:
close_q = '''
SELECT player_id, date, over_under, number, {}
FROM closing_odds
WHERE date BETWEEN '{}' AND '{}'
'''.format(', '.join(BOOKS), SEASON_START, SEASON_END)

closing = pd.read_sql(close_q, con)
closing['date'] = pd.to_datetime(closing['date'])

clv = matched.merge(closing, on=['player_id', 'date', 'over_under', 'number'],
                    how='left', suffixes=('', '_close'))

In [ ]:
# Use the book you actually bet, not the best close, or you will flatter yourself.
clv['close_odds'] = clv.apply(lambda r: r.get('{}_close'.format(r['book']), np.nan), axis=1)
clv = clv.dropna(subset=['close_odds'])

clv['close_dec'] = american_to_decimal(clv['close_odds'])
clv['clv'] = clv['close_dec'] / clv['dec_odds'] - 1

print('mean CLV:  {:.2%}'.format(clv['clv'].mean()))
print('beat close: {:.1%} of bets'.format((clv['clv'] < 0).mean()))
print()
print(clv.groupby(['over_under', 'implied_bucket'])['clv'].agg(['size', 'mean']))

In [ ]:
# Does CLV predict ROI at the bet level? It should, weakly but positively.
clv['ret'] = clv['profit'] / clv['stake']
X = sm.add_constant(clv[['clv']])
print(sm.OLS(clv['ret'], X).fit(cov_type='cluster',
      cov_kwds={'groups': clv['date']}).summary())

## 8. Retrain

Roll the window: drop the oldest season, add the one just completed. Refit and compare on a
holdout the incumbent model never saw.

The comparison that matters is **new model vs. incumbent, on identical holdout data**. A Brier
improvement against a differently-constructed holdout is not evidence.

In [ ]:
train_q = open('nba/data/sql/threeRunQ.sql').read()
frame = pd.read_sql(train_q, con)
frame['game_date'] = pd.to_datetime(frame['game_date'])
print(frame.shape)
frame['season'].value_counts().sort_index()

In [ ]:
TRAIN_SEASONS = ['2023-24', '2024-25']
TEST_SEASONS = ['2025-26']

FEATURES = [
    # incumbent feature set — keep this list authoritative and version it
]

train = frame[frame['season'].isin(TRAIN_SEASONS)].dropna(subset=FEATURES + ['threesMade'])
test = frame[frame['season'].isin(TEST_SEASONS)].dropna(subset=FEATURES + ['threesMade'])

y_train = train['threesMade'].clip(upper=10).astype(int)
y_test = test['threesMade'].clip(upper=10).astype(int)

X_train = sm.add_constant(train[FEATURES])
X_test = sm.add_constant(test[FEATURES])

In [ ]:
def multiclass_brier(P, y, n_class=11):
    Y = np.zeros((len(y), n_class))
    Y[np.arange(len(y)), y] = 1
    return ((P - Y) ** 2).sum(axis=1).mean()


base_model = sm.MNLogit(y_train, X_train).fit(method='bfgs', maxiter=1000)
P_base = base_model.predict(X_test).to_numpy()
brier_base = multiclass_brier(P_base, y_test.to_numpy())
print('incumbent holdout Brier: {:.4f}'.format(brier_base))

In [ ]:
# Candidate: defensive shot profile conceded (wide-open 3PA share vs opponent's usual diet).
NEW_FEATURES = FEATURES + ['def_profile_conceded']

train2 = frame[frame['season'].isin(TRAIN_SEASONS)].dropna(subset=NEW_FEATURES + ['threesMade'])
test2 = frame[frame['season'].isin(TEST_SEASONS)].dropna(subset=NEW_FEATURES + ['threesMade'])

m2 = sm.MNLogit(train2['threesMade'].clip(upper=10).astype(int),
                sm.add_constant(train2[NEW_FEATURES])).fit(method='bfgs', maxiter=1000)
P2 = m2.predict(sm.add_constant(test2[NEW_FEATURES])).to_numpy()
brier_new = multiclass_brier(P2, test2['threesMade'].clip(upper=10).astype(int).to_numpy())

print('incumbent: {:.4f}'.format(brier_base))
print('with def:  {:.4f}'.format(brier_new))
print('delta:     {:+.4f}'.format(brier_new - brier_base))

In [ ]:
# Paired bootstrap on the Brier delta. A raw improvement of 0.002 on ~20k rows may or may not
# clear the noise floor; this tells you which. Cluster by slate date, same reasoning as above.
common = test2.index.intersection(test.index)
loss_base = ((P_base - np.eye(11)[y_test.to_numpy()]) ** 2).sum(axis=1)
loss_new = ((P2 - np.eye(11)[test2['threesMade'].clip(upper=10).astype(int).to_numpy()]) ** 2).sum(axis=1)

# align on common rows before differencing — placeholder, adapt to your index handling
d = pd.Series(loss_new - loss_base, index=test2.index)
dates = test2['game_date']

rng = np.random.default_rng(0)
uniq = dates.unique()
deltas = []
for _ in range(2000):
    draw = rng.choice(uniq, size=len(uniq), replace=True)
    deltas.append(np.concatenate([d[dates == x].to_numpy() for x in draw]).mean())

deltas = np.array(deltas)
print('mean Brier delta: {:+.5f}  CI [{:+.5f}, {:+.5f}]'.format(
    deltas.mean(), np.percentile(deltas, 2.5), np.percentile(deltas, 97.5)))

In [ ]:
# Halflife grid for the defensive rolling metric, evaluated through holdout Brier in the
# full model. Center the grid on the reliability-curve result rather than sweeping blind.
HALFLIVES = [5, 10, 15, 20, 30, 45]

results = []
for h in HALFLIVES:
    # TODO: rebuild def_profile_conceded at halflife h, refit, score
    results.append({'halflife': h, 'brier': np.nan})

pd.DataFrame(results)

In [ ]:
import pickle
from datetime import datetime

artifact = {
    'fitted_at': datetime.now().isoformat(),
    'train_seasons': TRAIN_SEASONS,
    'features': NEW_FEATURES,
    'holdout_brier': brier_new,
    'model': m2,
}
# with open('models/mnlogit_{}.pkl'.format(SEASON), 'wb') as f:
#     pickle.dump(artifact, f)

## 9. Decisions log

Fill this in at the end and carry it into next season. Written before the season starts, these
are hypotheses; written after, they are results. Do not blur the two.

| # | Question | Evidence | Decision | Revisit when |
|---|----------|----------|----------|--------------|
| 1 | Is the realized edge distinguishable from zero? | §4 CI, §7 CLV | | |
| 2 | Do the pre-registered edge zones replicate? | §4 zone_ci | | |
| 3 | Static or dynamic bankroll for next season? | §6 counterfactuals | | |
| 4 | Cap on single-bet fraction? | §6a, §6b | | |
| 5 | Ship the defensive shot-profile feature? | §8 paired bootstrap | | |
| 6 | Chosen halflife for the defensive metric | §8 grid | | |
| 7 | Headline ROI definition going forward | §3a | | |

**Pre-register for next season, before any bets are placed:**

- The exact bet-selection rule (implied-probability bounds, minimum edge, minimum model prob).
- The sizing rule and any cap.
- The stopping rule.

Anything decided mid-season is a new hypothesis and should be tracked separately, not folded
into the season's headline number.